# 02 – Feature engineering

Placeholder: build templates, embeddings, or vectorized features from parsed logs.

In [1]:
from pathlib import Path
import re, hashlib, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
CONFIG = {
    "window_size": 20,
    "max_rows": None,      # None = full file; or set e.g. 200000 for faster iteration
}

WINDOW_SIZE = int(CONFIG["window_size"])
MAX_ROWS = CONFIG["max_rows"]

In [3]:
def find_repo_root(start: Path) -> Path:
    """
    Walk upward until we find a likely repo root.
    Works when you run from notebooks/ or repo root.
    """
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "requirements.txt").exists() or (p / "README.md").exists():
            return p
    return start

In [4]:
REPO_ROOT = find_repo_root(Path.cwd())

# Support both: machine_learning/ and "machine learning"/
ML_CANDIDATES = [
    REPO_ROOT / "machine_learning",
    REPO_ROOT / "machine learning",
]
ML_DIR = next((p for p in ML_CANDIDATES if p.exists()), ML_CANDIDATES[0])

RAW_DIR = ML_DIR / "data" / "raw"
PROCESSED_DIR = ML_DIR / "data" / "processed"
FEATURES_DIR = ML_DIR / "features"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

STRUCTURED_PATH = PROCESSED_DIR / "hdfs_structured.parquet"  # from File 01
TEMPL_COUNTS_PATH = PROCESSED_DIR / "template_counts.csv"
# Use canonical names expected by 03_train_isoforest and 04_infer_and_score
X_PATH = FEATURES_DIR / "hdfs_window_features.npz"
ENCODER_PATH = FEATURES_DIR / "hdfs_encoder.json"

print("REPO_ROOT:", REPO_ROOT)
print("ML_DIR:", ML_DIR)
print("STRUCTURED_PATH:", STRUCTURED_PATH)
print("WINDOW_SIZE:", WINDOW_SIZE)

if not STRUCTURED_PATH.exists():
    raise FileNotFoundError(
        f"Missing structured parquet: {STRUCTURED_PATH}\n"
        f"Run 01_parse_hdfs.ipynb first (it must save hdfs_structured.parquet)."
    )

REPO_ROOT: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent
ML_DIR: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning
STRUCTURED_PATH: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/processed/hdfs_structured.parquet
WINDOW_SIZE: 20


In [5]:
df = pd.read_parquet(STRUCTURED_PATH)
if MAX_ROWS is not None:
    df = df.head(int(MAX_ROWS)).copy()

required_cols = {"message"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Structured parquet missing columns: {missing}. Found columns: {list(df.columns)}")

print("Loaded df shape:", df.shape)

Loaded df shape: (2000, 9)


In [6]:
RE_BLOCK = re.compile(r"\bblk_-?\d+\b")
RE_IPPORT = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?\b")
RE_NUMBER = re.compile(r"\b\d+\b")
RE_HEX = re.compile(r"\b0x[0-9a-fA-F]+\b")

def to_template(msg: str) -> str:
    s = str(msg)

    # Order matters: specific first
    s = RE_BLOCK.sub("<BLK>", s)
    s = RE_IPPORT.sub("<IP>", s)
    s = RE_HEX.sub("<HEX>", s)

    # Collapse standalone numbers last
    s = RE_NUMBER.sub("<NUM>", s)

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [7]:
def template_id(tpl: str, n=10) -> str:
    h = hashlib.sha1(tpl.encode("utf-8")).hexdigest()
    return f"tpl_{h[:n]}"

df["template"] = df["message"].astype(str).apply(to_template)
df["template_id"] = df["template"].apply(template_id)

print("Unique templates:", df["template_id"].nunique())
display(df[["message", "template", "template_id"]].head(10))

Unique templates: 192


,message,template,template_id
0,PacketResponder 1 for block blk_38865049064139...,PacketResponder <NUM> for block <BLK> terminating,tpl_5dbdaf1727
1,PacketResponder 0 for block blk_-6952295868487...,PacketResponder <NUM> for block <BLK> terminating,tpl_5dbdaf1727
2,BLOCK* NameSystem.addStoredBlock: blockMap upd...,BLOCK* NameSystem.addStoredBlock: blockMap upd...,tpl_632f57fcfe
3,PacketResponder 2 for block blk_82291938032499...,PacketResponder <NUM> for block <BLK> terminating,tpl_5dbdaf1727
4,PacketResponder 2 for block blk_-6670958622368...,PacketResponder <NUM> for block <BLK> terminating,tpl_5dbdaf1727
5,BLOCK* NameSystem.addStoredBlock: blockMap upd...,BLOCK* NameSystem.addStoredBlock: blockMap upd...,tpl_632f57fcfe
6,BLOCK* NameSystem.addStoredBlock: blockMap upd...,BLOCK* NameSystem.addStoredBlock: blockMap upd...,tpl_632f57fcfe
7,BLOCK* NameSystem.addStoredBlock: blockMap upd...,BLOCK* NameSystem.addStoredBlock: blockMap upd...,tpl_632f57fcfe
8,PacketResponder 2 for block blk_57249283928729...,PacketResponder <NUM> for block <BLK> terminating,tpl_5dbdaf1727
9,Received block blk_3587508140051953248 of size...,Received block <BLK> of size <NUM> from /<IP>,tpl_be685b10cf


In [8]:
tpl_counts = (
    df.groupby(["template_id", "template"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)
display(tpl_counts.head(20))

,template_id,template,count
70,tpl_632f57fcfe,BLOCK* NameSystem.addStoredBlock: blockMap upd...,314
65,tpl_5dbdaf1727,PacketResponder <NUM> for block <BLK> terminating,311
190,tpl_fe28e3c886,Receiving block <BLK> src: /<IP> dest: /<IP>,292
143,tpl_be685b10cf,Received block <BLK> of size <NUM> from /<IP>,292
180,tpl_f8d16a8575,BLOCK* NameSystem.delete: <BLK> is added to in...,224
185,tpl_fae81fc90c,<IP>:Got exception while serving <BLK> to /<IP>:,80
139,tpl_b748fd0d08,<IP> Served block <BLK> to /<IP>,80
69,tpl_62ca1ae977,Verification succeeded for <BLK>,20
46,tpl_3e56d2ab53,Deleting block <BLK> file /mnt/hadoop/dfs/data...,15
5,tpl_0f39ca25aa,Deleting block <BLK> file /mnt/hadoop/dfs/data...,11


In [9]:
le = LabelEncoder()
df["template_idx"] = le.fit_transform(df["template_id"])
num_templates = len(le.classes_)

print("Number of unique templates:", num_templates)

Number of unique templates: 192


In [10]:
template_sequence = df["template_idx"].values
if len(template_sequence) < WINDOW_SIZE:
    raise ValueError(f"Not enough rows ({len(template_sequence)}) for WINDOW_SIZE={WINDOW_SIZE}")

windows = []
for i in range(len(template_sequence) - WINDOW_SIZE + 1):
    windows.append(template_sequence[i:i+WINDOW_SIZE])
windows = np.array(windows, dtype=np.int32)

print("Total windows:", windows.shape[0])
print("Window shape:", windows.shape)


Total windows: 1981
Window shape: (1981, 20)


In [11]:
X = np.zeros((windows.shape[0], num_templates), dtype=np.float32)

for i, window in enumerate(windows):
    # count occurrences in this window
    # faster than nested python loops
    vals, counts = np.unique(window, return_counts=True)
    X[i, vals] = counts

print("Feature matrix shape (X):", X.shape)

Feature matrix shape (X): (1981, 192)


In [12]:
df.to_parquet(STRUCTURED_PATH, index=False)

tpl_counts.to_csv(TEMPL_COUNTS_PATH, index=False)

np.savez_compressed(X_PATH, X=X)

with open(ENCODER_PATH, "w") as f:
    json.dump({
        "classes_": le.classes_.tolist(),   # idx -> template_id
        "window_size": WINDOW_SIZE,
        "num_templates": int(num_templates),
        "num_windows": int(X.shape[0]),
    }, f, indent=2)

print("\nSaved:")
print(" -", STRUCTURED_PATH)
print(" -", TEMPL_COUNTS_PATH)
print(" -", X_PATH)
print(" -", ENCODER_PATH)


Saved:
 - /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/processed/hdfs_structured.parquet
 - /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/processed/template_counts.csv
 - /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/features/hdfs_window_features.npz
 - /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/features/hdfs_encoder.json
